# 6.1 Introduction to Multi-Agent Systems

## Background

A single drone has limited capabilities. Multiple drones or "swarms" can cooperate to accomplish tasks.
Multi-drone systems offer significant advantages in efficiency, task coverage, and robustness.
Cooperation enables capabilities beyond what any single agent can achieve.

![drone_cluster.png](img/drone_cluster.png)

## LLM-Based Multi-Agent Systems
This course explores using Large Language Model (LLM) based agents to control multiple drones. We will implement and verify:
- Building and deploying multi-drone agents in the high-fidelity AirSim simulator.

For multi-agent systems, whether drones or computing agents, the following challenges must be addressed:

- Task Allocation and Decomposition: How to break down high-level task objectives (e.g., "search this area") into executable sub-tasks (e.g., "drone 1 searches zone A", "drone 2 searches zone B"), and how to assign these tasks to agents.
- Communication: What information do agents need to share? How to build communication protocols and hierarchies?
- Decentralized Decision-Making and Consensus: When not all agents can be under centralized control, how do agents make decisions based on local information? How to reach consensus?
- Path Planning and Deconfliction: This is a critical challenge. Each drone needs to plan its path to the target while avoiding collisions with other drones and obstacles in the environment.


## Multi-Agent Collaboration Patterns

Connecting multiple LLM Agents together requires coordination and communication. The key challenge is how to manage information flow and control between agents.

![drone_level.jpg](img/drone_level.jpg)

### Supervisor Pattern (Centralized)

- **Concept**: Also known as the hub-and-spoke model. A "Supervisor" agent receives the high-level task objective, decomposes it into sub-tasks, and assigns them to multiple "Worker" agents. Each worker agent controls a drone and executes its assigned task (e.g., searching an area, collecting data). After completing their tasks, workers report results back. The supervisor evaluates task completion and decides next steps.
- **Implementation (with LangGraph)**: This can be represented using a graph structure. Agents are nodes in the graph, and transitions between nodes are determined by the current state. In LangGraph, conditional edges can be used to implement routing logic. This pattern is well-suited for parallel task scenarios such as search operations or large-area coverage.

### Handoffs and Hierarchical Teams

- **Concept**: When tasks require specialization or sequential processing, agents can hand off control to each other.

**Control Handoffs**: Agents can transfer control between each other. For example, a "search" agent that finds a target can hand off the target coordinates and control to a "close-range inspection" agent with a high-resolution camera.

**Hierarchical Teams**: High-level agents can manage multiple sub-teams. For example, a "task coordinator" agent manages a "search team" and an "inspection team". Each team manages the drones that execute the actual tasks.

- **Implementation (with AutoGen)**: AutoGen's "conversable" agents simulate this pattern well. The GroupChatManager can coordinate multiple agents. Based on conversation history (i.e., the message records between agents), it determines which agent should speak next. This enables flexible, dynamic collaboration.

## Multi-Agent Frameworks

Currently there are several major frameworks that support multi-agent systems. The most prominent ones are Microsoft AutoGen, LangChain's LangGraph, and CrewAI.

Let's analyze their characteristics and suitability for drone simulation:

# Microsoft AutoGen: Conversation-Driven

![autogen.png](img/autogen.png)


## Core Philosophy
AutoGen: Multi-agent collaboration through conversation. It provides a graph-based task execution framework and offers flexible tools to create "conversable" agents. These agents can be LLMs, code executors, or tools that collaborate through message passing.

- **Core idea**: Multi-agent collaboration through conversation.
- **Goal**: Provide flexible tools to create "conversable" agents.
- **Agent types**: LLM, code executor, or tool.
- **Collaboration method**: Through message passing and conversation.

## Collaboration Mechanism
AutoGen's core mechanism is "Group Chat". The GroupChatManager coordinates conversations between multiple agents. The key is determining which agent should speak next based on conversation history and context. AutoGen supports various conversation patterns including:

### Sequential Chat
Tasks are processed by agents in a fixed sequential order.

### Nested Chat
Agents can be composed hierarchically, with one agent calling another as a sub-routine.

### Hierarchical Chat
Agents have management relationships, e.g., a "supervisor" agent managing multiple "worker" agents.

## Advantages
AutoGen excels when the solution path is not predetermined and requires exploration through conversation. It supports iterative reasoning where agents can discuss, debate, and refine solutions. Additionally, AutoGen is well-suited for scenarios requiring dynamic collaboration and exploration of complex problems.

- **Exploration**: Best for tasks where the solution path is not predetermined.
- **Iterative reasoning**: Supports back-and-forth discussion between agents.
- **Dynamic collaboration**: Well-suited for complex, open-ended scenarios.

Link: https://www.microsoft.com/en-us/research/project/autogen/

# LangChain LangGraph: Stateful Graph Orchestration

![agent_workflow.avif](img/agent_workflow.avif)


## Core Philosophy
LangGraph is built on top of LangChain for building deterministic, controllable multi-agent workflows. Its core concept is the **Stateful Graph**.
- In this graph, **Nodes** represent agents or functions.
- **Edges** define the transitions and control flow between nodes.
- This approach provides precise control over process flow, routing, and execution order.

## Collaboration Mechanism
In LangGraph, agents communicate through a **shared state object**.
- After each node executes, it updates this state.
- Subsequent nodes read from the state, enabling data passing and control flow.

Control flow is managed through:

### 1. Tool Calling
- **Type**: Centralized control.
- **Description**: A "Supervisor" agent manages the overall control flow.
- **Mechanism**: The supervisor agent calls other agents as "tools":
  - Receives a task
  - Calls a tool (i.e., another agent)
  - Passes parameters and receives results
- Best used for structured, deterministic process scenarios.

### 2. Handoffs
- **Type**: Decentralized control.
- **Description**: The current agent decides to transfer control to the next agent based on the task.
- **Mechanism**: Flexible routing and dynamic agent selection.
- Best used for tasks requiring specialization and sequential processing.

## Advantages
LangGraph excels at building **highly controllable, observable** multi-agent systems:

- Precise control: Supports branching, conditional logic, and loops.
- High observability: Graph-based state enables easy debugging and monitoring.
- Determinism: Deterministic execution is ideal for scenarios requiring high reliability.
- Flexibility: Nodes can be reused, composed, iterated, and extended.

> Summary: LangGraph provides a powerful "state + graph" paradigm for building controllable, observable Agent applications.

Link: https://www.langchain.com/langgraph

# CrewAI: Task-Driven Execution

![crewai.png](img/crewai.png)


![crewai2.png](img/crewai2.png)

## Core Philosophy
CrewAI focuses on rapidly building multi-agent applications through high-level abstractions. Agents are defined by their roles and responsibilities rather than low-level message passing. You define each agent's Role, Goal, and Backstory. These agents form a "Crew" that works together like a real team.

- **Core idea**: Agents defined by roles and responsibilities.
- **Approach**: Define each agent's role, goal, and backstory.
- **Organization**: Agents form a "Crew" for high-level collaboration.

## Collaboration Mechanism
CrewAI focuses on goals and tasks. Within a "Crew", you define Sequential or Parallel processing flows. Task Delegation is a key feature where agents can autonomously delegate tasks to other agents better suited to complete them. This provides a clean, structured workflow.

### Task Delegation
- **Description**: Agents can delegate tasks to other agents better suited for the work.
- **Process types**:
  - **Sequential Process**: Tasks are executed one after another in order.
  - **Parallel Process**: Multiple tasks execute simultaneously.

## Advantages
CrewAI's greatest advantage is its simplicity and low learning curve, making it a rapid prototyping tool. For tasks that can be mapped to "team roles" with clear sequential steps (e.g., research -> analyze data -> generate report), CrewAI significantly reduces complexity and makes the technology accessible.

- **Simplicity**: Easy to understand and use, rapid setup.
- **Low learning curve**: Minimal boilerplate code needed.
- **Structured workflows**: Clear task steps and role assignments.
- **Use cases**: Best for sequential task pipelines, research, data analysis, and content generation.

> Summary: CrewAI provides an intuitive, role-based approach to multi-agent systems, ideal for structured, step-by-step task execution.

Link: https://www.crewai.com/

## Analysis for Drone Applications

For drone applications, let's analyze the advantages of each framework:

- **LangGraph** provides precise control and observability. If the task is to build a drone system with a deterministic process in a simulation environment, it is the best choice. Its state-based approach and "tool calling" pattern work well for structured drone operations. It can implement both "supervisor" and "handoff" patterns effectively.
- **CrewAI** provides simplicity and rapid prototyping. For quickly verifying role-based drone workflows (e.g., "search" -> "analyze" -> "report"), it offers high efficiency. However, its limited control granularity may not fully support the real-time, dynamic decision-making needed in actual drone tasks.
- **AutoGen** has unique advantages in exploration and dynamic conversation. Its strength lies in the "conversable" agent paradigm, which enables simulation and implementation of complex agent interactions. For drones operating in dynamic environments, agents may need real-time communication and negotiation - for example, when drones encounter obstacles, they need to notify other drones through conversation. AutoGen's GroupChat mechanism naturally supports this type of interaction.

In summary, for structured multi-drone applications, LangGraph or CrewAI's graph-based approach works best. For exploratory applications requiring dynamic agent interaction, AutoGen is more suitable.

For this course on task-driven drone control, LangGraph's deterministic control flow is ideal. However, AutoGen's conversational approach is valuable for scenarios requiring negotiation between agents.

**This course will primarily use LangGraph to build multi-drone Agent scenarios based on AirSim.**